# Breast cancer dataloader

Original experiment from [Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow](https://github.com/Muhammad-Huzifa/Neural-Networks-and-Deep-Learning-Using-Pytorch-and-Tensor-Flow/blob/dfa17b2b298631b10e5904b2f1c39ad64d424bc1/Pytorch/Dataset%20and%20Dataloader%20using%20Breast%20Cancer.ipynb). Run from a fresh kernel. See [dataset and environment instructions](../../../docs/DATASETS.md).

Framework training, downloaded datasets, and custom image inputs require the dependencies and resources described in the collection README. This migration has checked syntax and paths; it has not repeated every training experiment.

In [ ]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements/base.txt").is_file())
TABULAR = ROOT / "data/tabular"
IMAGES = ROOT / "data/images"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yasserh/breast-cancer-dataset")

print("Path to dataset files:", path)

In [ ]:
import os
for file in os.listdir(path):
    print(file)

In [ ]:
df = pd.read_csv(path+"/breast-cancer.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.drop(columns=['id'],inplace=True)
df = pd.DataFrame(df)  # Convert list to DataFrame (if applicable)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2, random_state=42)

In [ ]:
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)

In [ ]:
encodor =LabelEncoder()
y_train = encodor.fit_transform(y_train)
y_test = encodor.transform(y_test)

In [ ]:
y_train[0:10]

In [ ]:
X_train_tesnor = torch.tensor(X_train, dtype=torch.float32)
X_test_tnesor = torch.tensor(X_test, dtype=torch.float32)
y_train_tesnor = torch.tensor(y_train, dtype=torch.float32)
y_test_tesnor = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
from torch.utils.data import DataLoader, Dataset
class CostumDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
train_dataset = CostumDataset(X_train_tesnor, y_train_tesnor)
test_dataset = CostumDataset(X_test_tnesor, y_test_tesnor)

In [ ]:
train_dataset[10]

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch.nn as nn
class MysimpleNN(nn.Module):
  def __init__(self , num_feature):
    super().__init__()
    self.linear = nn.Linear(num_feature,5)
    self.relu = nn.ReLU()
    self.linear2 = nn.Linear(5,1)
    self.sigmoid = nn.Sigmoid()
  def forward(self,features):
    out = self.linear(features)
    out = self.relu(out)
    out = self.linear2(out)
    out = self.sigmoid(out)
    return out

In [ ]:
lr  = 0.001
epochs = 100

In [ ]:
model = MysimpleNN(X_train_tesnor.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.BCELoss()
loss_function = nn.BCELoss()

In [ ]:
for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:
    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_labels.view(-1, 1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

In [ ]:
model
